In [22]:
#!pip install pdfminer.six   --> extracts text from PDF files
#!pip install spacy
#!python -m spacy download en_core_web_sm   --> NLP model for entity recognition/name extraction

In [23]:
from pdfminer.high_level import extract_text    # For extracting text from PDF files
import os                             # For handling file paths

def extract_text_from_pdf(pdf_path):
    return extract_text(pdf_path)       # Extract text from the pdf file into sample text

sample_text = extract_text_from_pdf('My Resume - SarvaGupta.pdf')       # Extract text from the specified PDF file
print(sample_text[:1000])       # Print the first 1000 characters to verify text extraction

Sarva Gupta 

Computer Science student with a strong foundation in software development, algorithms, artiﬁcial intelligence, and data-driven systems, built through
undergraduate and graduate coursework. Experienced in delivering technical, data-oriented, and full-stack projects, collaborating across teams, and
communicating complex ideas eﬀectively. 

sarva.gupta.3@gmail.com 

602-223-1850 

Phoenix, AZ, United States 

linkedin.com/in/sarva-gupta-42826b256 

EDUCATION 

SKILLS 

Master of Science in Computer Science 
Arizona State University 
01/2025 - 05/2026,  

Python 

Java 

C# 

JavaScript 

SQL 

Tempe, AZ 

ASP .NET 

Rest APIs 

HTML 

CSS 

Git 

Bachelor of Science in Computer Science 
ASU (Barrett, the Honors College) 
08/2021 - 12/2024,  

Tempe, AZ, 3.50 

Visual Studio 

Firebase 

Linux 

Figma 

React 

Taiga 

Agile 

UI/UX Design 

High School (with Honors) 
BASIS Phoenix 
08/2017 - 05/2021,  

Phoenix, AZ, 3.83 

PROJECTS 

WORK EXPERIENCE 

Undergraduate Teaching 

In [24]:
import re       # For regular expressions to clean the text

def clean_text(text):
    text = re.sub(r'\n+', '\n', text)  # Replace multiple newlines with a single newline   
    # \n = newline, + = 1/more so \n+ = 1/more newlines
    text = re.sub(r' +', ' ', text)  # Replace multiple spaces with a single space
        # ' ' = space, + = 1/more so ' '+ = 1/more spaces
    return text.strip()

cleaned_text = clean_text(sample_text)      # Clean the extracted text to make it easier to parse and use
print(cleaned_text)     # Print the cleaned text to verify the cleaning process

Sarva Gupta 
Computer Science student with a strong foundation in software development, algorithms, artiﬁcial intelligence, and data-driven systems, built through
undergraduate and graduate coursework. Experienced in delivering technical, data-oriented, and full-stack projects, collaborating across teams, and
communicating complex ideas eﬀectively. 
sarva.gupta.3@gmail.com 
602-223-1850 
Phoenix, AZ, United States 
linkedin.com/in/sarva-gupta-42826b256 
EDUCATION 
SKILLS 
Master of Science in Computer Science 
Arizona State University 
01/2025 - 05/2026, 
Python 
Java 
C# 
JavaScript 
SQL 
Tempe, AZ 
ASP .NET 
Rest APIs 
HTML 
CSS 
Git 
Bachelor of Science in Computer Science 
ASU (Barrett, the Honors College) 
08/2021 - 12/2024, 
Tempe, AZ, 3.50 
Visual Studio 
Firebase 
Linux 
Figma 
React 
Taiga 
Agile 
UI/UX Design 
High School (with Honors) 
BASIS Phoenix 
08/2017 - 05/2021, 
Phoenix, AZ, 3.83 
PROJECTS 
WORK EXPERIENCE 
Undergraduate Teaching Assistant - Intro to
Informatics 
Ira

In [25]:
import re

def extract_email(text):        # Extract email addresses using a regular expression
    match = re.search(r'\S+@\S+', text)
    return match.group(0) if match else None


def extract_phone_number(text):         # Extract and normalize phone numbers
    # Find any US-style phone number with optional country code and ANY separators
    pattern = r'(\+?\d{1,3}[\s().\-]*)?(\(?\d{3}\)?[\s().\-]*\d{3}[\s().\-]*\d{4})'
    match = re.search(pattern, text)

    if not match:
        return None

    # Remove everything except digits
    digits_only = re.sub(r'\D', '', match.group(0))

    # If a country code exists (e.g., +1), keep the last 10 digits
    if len(digits_only) < 10:
        return None
    digits_only = digits_only[-10:]

    # Format consistently
    return f"{digits_only[:3]}-{digits_only[3:6]}-{digits_only[6:]}"


email = extract_email(cleaned_text)
phone_number = extract_phone_number(cleaned_text)

print(f"Extracted Email: {email}")
print(f"Extracted Phone Number: {phone_number}")


Extracted Email: sarva.gupta.3@gmail.com
Extracted Phone Number: 602-223-1850


In [26]:
import re
import spacy

nlp = spacy.load('en_core_web_sm')      # Load the spaCy model for English to use its named entity recognition capabilities

def extract_name(text):     # Extract the candidate's name using heuristics and spaCy's named entity recognition
    lines = [ln.strip() for ln in text.split('\n') if ln.strip()]
    top_lines = lines[:15]

    bad_words = {       # Words that commonly appear in resumes but are not names
        'resume', 'curriculum vitae', 'cv', 'email', 'phone', 'address',
        'linkedin', 'github', 'portfolio', 'objective', 'summary',
        'education', 'experience', 'skills', 'projects'
    }

    # ⭐ Prefer first line if it looks like a name
    first_line = top_lines[0]
    if re.fullmatch(r"[A-Za-z][A-Za-z.\-'\s]{1,80}", first_line):
        if not any(w in first_line.lower() for w in bad_words):
            return first_line

    candidates = []

    for ln in top_lines:
        ln_clean = re.sub(r'\s+', ' ', ln.strip())      # Clean up the line by removing extra spaces and trimming it

        if any(w in ln_clean.lower() for w in bad_words):       # If the line contains any bad words, skip it
            continue

        if not re.fullmatch(r"[A-Za-z][A-Za-z.\-'\s]{1,80}", ln_clean):     # If the line doesn't look like a name, skip it
            continue

        words = ln_clean.replace('.', '').split()     # Remove periods and split the line into words to check the word count
        if 2 <= len(words) <= 4:
            candidates.append(ln_clean)

    if candidates:
        return candidates[0]  # first valid candidate wins

    # Fallback: spaCy on top chunk only
    doc = nlp("\n".join(top_lines))         # Use spaCy to analyze the top lines of the resume for named entities
    people = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
    return people[0] if people else None        # Return the first person entity found by spaCy if no candidates were found by heuristics


name = extract_name(cleaned_text)
print(f"Extracted Name: {name}")

Extracted Name: Sarva Gupta


In [27]:
# SKILL_SET = {'python', 'java', 'c++', 'machine learning', 'data analysis', 'sql', 'javascript'}

SKILL_SET = {
    # Programming Languages - excluding r, 
    'python', 'java', 'c', 'c++', 'c#', 'go', 'rust', 'scala',
    'javascript', 'typescript', 'matlab', 'bash', 'shell','r',

    # Web & Frontend
    'html', 'css', 'react', 'angular', 'vue', 'next.js', 'node.js',
    'express', 'rest', 'soap', 'graphql', 'bootstrap', 

    # Backend & Frameworks
    'django', 'flask', 'fastapi', 'spring', 'spring boot',
    '.net', 'asp.net',

    # Databases
    'sql', 'mysql', 'postgresql', 'sqlite', 'mongodb', 'redis',
    'nosql',

    # Data / ML / AI
    'machine learning', 'deep learning', 'artificial intelligence',
    'data analysis', 'data science', 'nlp', 'computer vision',
    'pandas', 'numpy', 'scikit-learn', 'tensorflow', 'pytorch',

    # Cloud & DevOps
    'aws', 'azure', 'gcp', 'docker', 'kubernetes',
    'ci/cd', 'jenkins', 'terraform',

    # Tools & Platforms
    'git', 'github', 'gitlab', 'bitbucket',
    'linux', 'unix', 'windows',

    # Software Engineering Practices
    'agile', 'scrum', 'kanban', 'oop', 'object oriented programming',
    'microservices', 'system design',

    # Testing
    'unit testing', 'integration testing', 'pytest', 'junit',

    # Visualization & UX
    'd3.js', 'tableau', 'power bi', 'figma', 'ux', 'ui',

    # Security
    'cybersecurity', 'authentication', 'authorization', 'oauth',
    'encryption'
}


def extract_skills(text, skills=SKILL_SET):
    found = [skill for skill in skills if skill.lower() in text.lower()]       # Check if each skill in the SKILL_SET is present in the text (case-insensitive) and collect the found skills in a list
    return list(set(found))  # Remove duplicates by converting to a set and back to a list

skillsList = extract_skills(cleaned_text)
print(f"Extracted Skills: ")
print(skillsList)
print(f"Total skills found:", len(skillsList))      # Print the total number of unique skills found in the resume

Extracted Skills: 
['d3.js', 'ux', 'asp.net', 'postgresql', 'express', 'c', 'html', 'rest', 'c#', 'linux', 'figma', 'encryption', 'ui', '.net', 'node.js', 'machine learning', 'go', 'git', 'agile', 'javascript', 'sql', 'python', 'css', 'java', 'react']
Total skills found: 25


In [28]:
EDUCATION_KEYWORDS = ['Associate', 'Associates', 'Bachelor', 'Master', 'Bachelors', 'Masters', 
                      'B.Sc', 'M.Sc', 'PhD', 'B.E', 'M.E.', 'B.S.', 'M.S.']

def extract_education(text):
    lines = text.split('\n')     # Split the text into lines to check for education-related keywords in each line
    education_info = []
    for line in lines:
        if any(keyword in line for keyword in EDUCATION_KEYWORDS):     # Check if any of the education-related keywords are present in the line
            education_info.append(line.strip())     # If a line contains any of the education keywords, add it to the education_info list after stripping leading/trailing whitespace
    return education_info

education_details = extract_education(cleaned_text)  # Extract education details from the cleaned text and store them in a list
print("Extracted Education Details:")
for edu in education_details:
    print("-", edu)    # Print each extracted education detail to verify the extraction process

Extracted Education Details:
- Master of Science in Computer Science
- Bachelor of Science in Computer Science


In [29]:
def extract_experience(text):  # Extract experience section lines by finding the Experience header and capturing everything underneath
    experience_header_keywords = [
        'work experience', 'experience', 'professional experience', 'research experience', 'technical experience', 'employment'
    ]
    
    # Common section headers that may appear AFTER experience (stop when we hit these)
    stop_section_keywords = [
        'project', 'projects', 'academic projects', 'education', 'skills', 'leadership',
        'activities', 'certifications', 'awards', 'publications', 'volunteer',
        'summary', 'objective', 'achievements', 'interests', 'references', 'languages', 'technical skills'
    ]
    
    lines = text.split('\n')
    experience_lines = []
    in_experience_section = False
    
    for line in lines:
        line_stripped = line.strip()
        line_lower = line_stripped.lower()
        
        # Start capturing when we hit an experience header line
        if (not in_experience_section) and line_stripped and any(k in line_lower for k in experience_header_keywords):
            in_experience_section = True
            continue  # skip the header line itself
        
        # If we're capturing and we hit another major section header, stop
        if in_experience_section and line_stripped and any(k == line_lower.rstrip(':') for k in stop_section_keywords):
            break
        
        # Keep everything underneath (including bullets), as long as it's not empty
        if in_experience_section and line_stripped:
            experience_lines.append(line_stripped)
    
    return experience_lines


experience_details = extract_experience(cleaned_text)  # Extract experience details from the cleaned text and store them in a list
print("Extracted Experience Details:")
for exp in experience_details:
    print("-", exp)  # Print each extracted experience detail to verify the extraction process


Extracted Experience Details:
- Undergraduate Teaching Assistant - Intro to
- Informatics
- Ira A. Fulton Schools of Engineering, ASU
- 08/2024 - 12/2024,
- Supported undergraduate coursework by reinforcing core computing concepts,
- assisting students with problem-solving, and contributing to a collaborative
- learning environment.
- Tempe, AZ
- Achievements/Tasks
- Supported 200+ students in Python-based coursework covering
- basics of Informatics, including data structures, algorithms, and
- problem-solving
- Led oﬃce hours and exam review sessions, clarifying algorithmic
- thinking and debugging strategies
- SWE Capstone – Full Stack Developer
- Ira A. Fulton Schools of Engineering, ASU
- 01/2024 - 12/2024,
- Tempe, AZ
- A year-long, team-based software engineering project with real-world stakeholders
- Achievements/Tasks
- Developed a web app using the PERN stack (PostgreSQL, Express,
- React, Node.js) to support mental-health peer communities
- Collaborated in an Agile team envir

In [30]:
import re

def extract_projects(text):
    project_header_keywords = ['projects', 'academic projects', 'project experience', 'selected projects']
    stop_section_keywords = [
        'experience', 'work experience', 'professional experience', 'employment',
        'education', 'skills', 'leadership', 'activities', 'certifications', 'awards',
        'publications', 'volunteer', 'summary', 'objective', 'achievements', 'interests', 'references', 'languages', 'technical skills'
    ]

    lines = [ln.strip() for ln in text.split('\n') if ln.strip()]
    projects_lines = []
    in_projects_section = False

    for line in lines:
        line_lower = line.lower().strip()

        # START: find the projects header (keyword style)
        if not in_projects_section and any(k == line_lower for k in project_header_keywords):
            in_projects_section = True
            continue  # skip the header line itself

        if in_projects_section:
            # STOP: only stop if we already captured some project content
            if projects_lines and any(k == line_lower for k in stop_section_keywords):
                break

            # ---- filter to prevent "work experience" column from getting sucked in ----
            # Skip likely job-title lines unless it's a bullet
            if (not line.startswith(('-', '•'))) and any(word in line_lower for word in [
                'assistant', 'intern', 'analyst', 'engineer', 'schools of', 'llc', 'co-founder', 'tempe', 'phoenix'
            ]):
                continue

            projects_lines.append(line)

    return projects_lines


project_details = extract_projects(cleaned_text)
print("Extracted Project Details:")
for proj in project_details:
    print("-", proj)


Extracted Project Details:
- WORK EXPERIENCE
- Informatics
- 08/2024 - 12/2024,
- Supported undergraduate coursework by reinforcing core computing concepts,
- assisting students with problem-solving, and contributing to a collaborative
- learning environment.
- Achievements/Tasks
- Supported 200+ students in Python-based coursework covering
- basics of Informatics, including data structures, algorithms, and
- problem-solving
- Led oﬃce hours and exam review sessions, clarifying algorithmic
- thinking and debugging strategies
- SWE Capstone – Full Stack Developer
- 01/2024 - 12/2024,
- Achievements/Tasks
- Developed a web app using the PERN stack (PostgreSQL, Express,
- React, Node.js) to support mental-health peer communities
- Collaborated in an Agile team environment, coordinating with
- designers, peer leaders, and a startup founder
- 10/2024 - 12/2025,
- Provided front-line academic and event support, assisting students and staﬀ in
- navigating programs, resources, and activities.


In [31]:
def parse_and_print_resume(text, filename="(single resume)"):   # Main function to parse the resume text and print the extracted information in a structured format for verification
    cleaned_text = clean_text(text)     # Clean the extracted text to make it easier to parse and use

    parsed_resume = {           # Create a structured dictionary to hold all the extracted information from the resume
        'name': extract_name(cleaned_text),
        'email': extract_email(cleaned_text),
        'phone_number': extract_phone_number(cleaned_text),
        'skills': extract_skills(cleaned_text),
        'education': extract_education(cleaned_text),
        'experience': extract_experience(cleaned_text),
        'projects': extract_projects(cleaned_text)
    }

    print("\n" + "=" * 80)              # Print a separator and the filename to clearly indicate the start of the parsed resume output
    print(f"PARSED RESUME: {filename}") 
    print("=" * 80)     # Print the cleaned text preview and all the extracted information in a structured format for verification

    print("\n[Cell 5] CLEANED TEXT (preview):")
    print(cleaned_text[:300])

    print("\n[Cell 6] NAME:")
    print(parsed_resume['name'])

    print("\n[Cell 7] EMAIL:")
    print(parsed_resume['email'])

    print("\n[Cell 8] PHONE NUMBER:")
    print(parsed_resume['phone_number'])

    print("\n[Cell 9] SKILLS:")
    print(parsed_resume['skills'])

    print("\n[Cell 10] EDUCATION:")
    for edu in parsed_resume['education']:
        print("-", edu)

    print("\n[Cell 11] EXPERIENCE:")
    for exp in parsed_resume['experience']:
        print("-", exp)

    print("\n[Cell 12] PROJECTS:")
    for proj in parsed_resume['projects']:
        print("-", proj)

    return parsed_resume        # Return the structured dictionary containing all the extracted information for potential use in further processing or exporting

In [32]:
import pandas as pd         
import os

def process_folder(folder_path):        # Process all PDF files in the specified folder, extract relevant information, and compile it into a DataFrame
    resumes_data = []

    for filename in sorted(os.listdir(folder_path)):        # Iterate through all files in the specified folder to find and process PDF files containing resumes
        if not filename.lower().endswith('.pdf'):
            continue

        pdf_path = os.path.join(folder_path, filename)      # Construct the full file path for the PDF file to be processed

        try:
            text = extract_text_from_pdf(pdf_path)
        except Exception as e:
            print(f"Failed to parse {filename}: {e}")       #  show error message if fails
            continue

        parsed_resume = parse_and_print_resume(text, filename)      # Parse the extracted text from the PDF file and print the structured information for verification
        parsed_resume['filename'] = filename  # helpful for CSV traceability
        resumes_data.append(parsed_resume)

    return pd.DataFrame(resumes_data)

df = process_folder('resumes')      # Process the folder containing the resume PDF files and create a DataFrame with the extracted information from all resumes
df.to_csv('parsed_resumes.csv', index=False)
df



PARSED RESUME: Aliyah St.Onge's Resume.pdf

[Cell 5] CLEANED TEXT (preview):
Aliyah St.Onge
(630) 488 - 7956 I aliyahstonge05@gmail.com I 1015 Schiedler Dr, Batavia, IL 60510
Education
Iowa State University Ames, IA 
Bachelor of Environmental Science Expected graduation: May 2029 
GPA: 3.78
Batavia High School Batavia, IL
GPA: 3.81 May 2025
Work Experience
Current Volunteer 

[Cell 6] NAME:
Aliyah St.Onge

[Cell 7] EMAIL:
aliyahstonge05@gmail.com

[Cell 8] PHONE NUMBER:
630-488-7956

[Cell 9] SKILLS:
['oop', 'go', 'rest', 'c']

[Cell 10] EDUCATION:
- Bachelor of Environmental Science Expected graduation: May 2029

[Cell 11] EXPERIENCE:
- Current Volunteer Server at the Moose Lodge Batavia, IL
- July 2025 - Present
- Ensure smooth service by regularly restocking to-go boxes, side dressings, and condiments such
- as ketchup and mustard
- Maintain cleanliness and organization by efficiently washing dishes, bussing tables, and sanitizing
- dining areas after guests leave
- Preparation of 

,name,email,phone_number,skills,education,experience,projects,filename
0,Aliyah St.Onge,aliyahstonge05@gmail.com,630-488-7956,"[oop, go, rest, c]",[Bachelor of Environmental Science Expected gr...,[Current Volunteer Server at the Moose Lodge B...,[],Aliyah St.Onge's Resume.pdf
1,ISAIAH MILKEY,isaiahmilkey@gmail.com,480-599-5646,"[scala, ux, deep learning, pandas, c, html, li...","[Bachelor of Science in Computer Science, Mast...","[Agentic AI Researcher, LENS Lab, Arizona Stat...",[],FEB2026_IsaiahMilkey_Resume.pdf
2,HRISIKA JAGDEEP,hrisika.jagdeep@gmail.com,925-719-2722,"[git, scala, encryption, junit, scrum, ui, ux,...",[B.S. Computer Science (Current)],[Team Leader & Co-Founder - Offline Digital Li...,"[Compiler - using C++, ● Developed a C++ progr...",Jagdeep_Hrisika_Resume.pdf
3,Sarva Gupta,sarva.gupta.3@gmail.com,602-223-1850,"[d3.js, ux, asp.net, postgresql, express, c, h...","[Master of Science in Computer Science, Bachel...","[Undergraduate Teaching Assistant - Intro to, ...","[WORK EXPERIENCE, Informatics, 08/2024 - 12/20...",My Resume - SarvaGupta.pdf
